# Pré-processamento v2 — features com dinâmica + backbones de emoção

Reescrita do pré-processamento. Diferenças-chave em relação ao v1 (que capava o resultado):

| | v1 | **v2** |
|---|---|---|
| Áudio (backbone) | `wav2vec2-base-960h` (ASR) | **`superb/wav2vec2-base-superb-er`** (afinado em emoção) |
| Pooling temporal | só `mean` (dinâmica perdida) | **`mean ⊕ std`** (statistics pooling → prosódia/expressão) |
| Frames de vídeo | 8 (duplicados p/ 16) | **16 reais** |
| Texto (backbone) | `bert-base` cru | encoder de **emoção/sentimento** + probs de classe |
| Rótulo | — | mantém `label_id` 0..6 (o modelo remapeia p/ valência ou 7 classes) |

Roda **uma vez** por idioma. Salva em `cache_{lang}_v2/multimodal/`. Idempotente (pula o que já existe).
Backbones em modo inference (`@torch.no_grad()`), congelados. Consumido pelo `model_v2.ipynb`.

## 1. Configuração

In [1]:
from pathlib import Path
import os, glob, torch

# idioma -> pasta de vídeos e modelo/idioma para transcrição/texto
LANG_CFG = {
    "en": {
        "videos": "../videos",
        "whisper_lang": "en",
        # encoder de emoção em inglês (7 emoções: anger/disgust/fear/joy/neutral/sadness/surprise)
        "text_model": "j-hartmann/emotion-english-distilroberta-base",
    },
    "pt": {
        "videos": "../videos_dub",
        "whisper_lang": "pt",
        # sentimento multilíngue (neg/neu/pos) — alinha com valência; robusto p/ PT
        "text_model": "cardiffnlp/twitter-xlm-roberta-base-sentiment",
    },
}

LANGS_TO_RUN = ["en", "pt"]      # processe um por vez se quiser economizar RAM
N_FRAMES     = 16                # frames reais por vídeo (VideoMAE espera 16)
FACE_MARGIN  = 0.20              # margem no recorte do rosto

# Backbones compartilhados entre idiomas (áudio é acústico; vídeo é visual)
AUDIO_MODEL  = "superb/wav2vec2-base-superb-er"
VIDEO_MODEL  = "MCG-NJU/videomae-base-finetuned-kinetics"

LABELS = {"Anger":0,"Disgust":1,"Fear":2,"Happy":3,"Neutral":4,"Sadness":5,"Surprised":6}

# Device: CUDA > DirectML > CPU. Inference-only, então LayerNorm forward funciona no DirectML.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    try:
        import torch_directml
        DEVICE = torch_directml.device()
    except Exception:
        DEVICE = torch.device("cpu")
# Whisper: decode autorregressivo é lento no DirectML (overhead por op) -> roda na CPU
# (ver célula de transcrição). 'base' é bem mais rápido que 'small' e suficiente p/ o texto.
WHISPER_MODEL = "openai/whisper-base" if DEVICE.type != "cuda" else "openai/whisper-large-v3"
print(f"Device: {DEVICE} | Whisper: {WHISPER_MODEL}")

def processed_dir(lang, sub): return f"../processed_{lang}/{sub}"
def cache_dir(lang):          return f"../cache_{lang}_v2/multimodal"

Device: cpu | Whisper: openai/whisper-base


In [2]:
import pandas as pd
from tqdm.auto import tqdm

def build_dataframe(lang):
    root = Path(LANG_CFG[lang]["videos"])
    rows = []
    for emotion, lid in LABELS.items():
        for v in sorted((root / emotion).glob("*.mp4")):
            rows.append({"video_path": str(v), "label": emotion, "label_id": lid, "sample_id": v.stem})
    df = pd.DataFrame(rows)
    print(f"[{lang}] {len(df)} vídeos | por classe: {df.groupby('label').size().to_dict()}")
    return df

c:\Users\luis-\Documents\codigos\academico\MultimodalEmotionReconizer\.venv-torch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Extração de áudio (ffmpeg → wav mono 16 kHz)

In [3]:
import ffmpeg

def extract_audio(video_path, out_path):
    (ffmpeg.input(str(video_path))
           .output(str(out_path), ac=1, ar=16000)
           .overwrite_output().run(quiet=True))

def run_audio_extraction(df, lang):
    out_dir = processed_dir(lang, "audio"); os.makedirs(out_dir, exist_ok=True)
    for r in tqdm(df.itertuples(), total=len(df), desc=f"[{lang}] áudio"):
        dst = f"{out_dir}/{r.sample_id}.wav"
        if not os.path.exists(dst):
            extract_audio(r.video_path, dst)

## 3. Transcrição (Silero VAD + Whisper)

In [4]:
import soundfile as sf
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

# Whisper roda na CPU: decode autorregressivo é lento no DirectML (overhead por op).
# 'base' é rápido e suficiente p/ o texto (modalidade fraca). Troque p/ 'small' se quiser + fidelidade.
WHISPER_ASR_MODEL  = "openai/whisper-base"
WHISPER_ASR_DEVICE = "cpu"

_whisper = {}   # reiniciado a cada execução desta célula -> recarrega com o modelo/CPU acima
def get_whisper():
    if "asr" not in _whisper:
        proc = AutoProcessor.from_pretrained(WHISPER_ASR_MODEL)
        mdl  = AutoModelForSpeechSeq2Seq.from_pretrained(WHISPER_ASR_MODEL).to(WHISPER_ASR_DEVICE)
        _whisper["asr"] = pipeline("automatic-speech-recognition", model=mdl,
                                   tokenizer=proc.tokenizer, feature_extractor=proc.feature_extractor,
                                   device=-1)
        vad_model, vad_utils = torch.hub.load("snakers4/silero-vad", "silero_vad",
                                              force_reload=False, trust_repo=True)
        _whisper["vad"] = vad_model
        _whisper["get_ts"], _, _, _, _whisper["collect"] = vad_utils
    return _whisper

def transcribe(wav_path, lang):
    w = get_whisper()
    data, _ = sf.read(wav_path)
    wav = torch.from_numpy(data).float()
    if wav.ndim > 1: wav = wav.mean(dim=1)
    ts = w["get_ts"](wav, w["vad"], sampling_rate=16000, threshold=0.5)
    if not ts:
        return ""
    speech = w["collect"](ts, wav)
    # SEM return_timestamps: clipes são curtos e já recortados pelo VAD -> elimina o loop de
    # alucinação (clipes de minutos). max_new_tokens limita fugas; no_repeat_ngram corta repetição.
    res = w["asr"](
        speech.numpy(),
        chunk_length_s=30,   # rede de segurança p/ eventuais áudios > 30s
        generate_kwargs={
            "task": "transcribe", "language": lang, "temperature": 0.0,
            "max_new_tokens": 96, "no_repeat_ngram_size": 3,
        },
    )
    return res["text"].strip()

def run_transcription(df, lang):
    out_dir = processed_dir(lang, "transcripts"); os.makedirs(out_dir, exist_ok=True)
    wlang = LANG_CFG[lang]["whisper_lang"]
    for r in tqdm(df.itertuples(), total=len(df), desc=f"[{lang}] transcrição"):
        dst = f"{out_dir}/{r.sample_id}.txt"
        if os.path.exists(dst): continue
        wav = f"{processed_dir(lang, 'audio')}/{r.sample_id}.wav"
        txt = transcribe(wav, wlang) if os.path.exists(wav) else ""
        with open(dst, "w", encoding="utf-8") as f: f.write(txt)

## 4. Rostos (RetinaFace → 16 frames por vídeo)

In [5]:
import cv2, urllib.request

# YuNet (OpenCV FaceDetectorYN): detector ONNX moderno, SEM TensorFlow/Keras (evita o crash do
# RetinaFace no TF 2.21/Keras 3). ~345KB, baixado 1x para ../models.
YUNET_URL = "https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx"

_det = {}
def get_detector():
    if "yn" not in _det:
        mdir = "../models"; os.makedirs(mdir, exist_ok=True)
        onnx = os.path.join(mdir, "face_detection_yunet_2023mar.onnx")
        if not os.path.exists(onnx):
            print("Baixando YuNet (detector de rosto)...")
            urllib.request.urlretrieve(YUNET_URL, onnx)
        _det["yn"] = cv2.FaceDetectorYN_create(onnx, "", (320, 320), 0.6, 0.3, 5000)
    return _det["yn"]

def _detect_box(frame, margin):
    """Caixa do maior rosto (com margem) num frame BGR, ou None."""
    det = get_detector()
    h, w = frame.shape[:2]
    det.setInputSize((w, h))
    _, faces = det.detect(frame)
    if faces is None or len(faces) == 0:
        return None
    x, y, fw, fh = max(faces, key=lambda r: r[-1])[:4]     # maior score
    mw, mh = int(fw * margin), int(fh * margin)
    return (max(0, int(x - mw)), max(0, int(y - mh)),
            min(w, int(x + fw + mw)), min(h, int(y + fh + mh)))

def extract_faces(video_path, out_dir, num_frames=N_FRAMES, margin=FACE_MARGIN):
    """Amostra num_frames uniformes, detecta o rosto UMA vez (rosto ~estático em clipe de fala)
    e reutiliza a caixa em todos os frames -> rápido e temporalmente estável. Salva 224x224."""
    os.makedirs(out_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release(); return
    want = {int(i * total / num_frames) for i in range(num_frames)}
    frames, i = [], 0
    while cap.isOpened() and len(frames) < num_frames:
        ret, fr = cap.read()
        if not ret: break
        if i in want: frames.append(fr)
        i += 1
    cap.release()
    if not frames: return
    while len(frames) < num_frames: frames.append(frames[-1])   # completa se faltou
    box = None
    for oi in (num_frames // 2, num_frames // 4, 3 * num_frames // 4, 0):
        box = _detect_box(frames[oi], margin)
        if box: break
    for k, fr in enumerate(frames):
        crop = fr[box[1]:box[3], box[0]:box[2]] if box else fr
        if crop.size == 0: crop = fr
        cv2.imwrite(f"{out_dir}/frame_{k:02d}.jpg", cv2.resize(crop, (224, 224)))

def run_faces(df, lang):
    base = processed_dir(lang, "faces"); os.makedirs(base, exist_ok=True)
    for r in tqdm(df.itertuples(), total=len(df), desc=f"[{lang}] rostos"):
        d = f"{base}/{r.sample_id}"
        if os.path.isdir(d) and len(glob.glob(f"{d}/*.jpg")) >= N_FRAMES: continue
        extract_faces(r.video_path, d)


## 5. Backbones de features (congelados) + statistics pooling

Todas as modalidades usam **mean ⊕ std** sobre a sequência (tokens/frames) → captura a dinâmica.
O texto acrescenta as **probabilidades de classe** do encoder de emoção/sentimento (sinal direto).

In [6]:
from PIL import Image
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoFeatureExtractor, Wav2Vec2ForSequenceClassification,
                          VideoMAEImageProcessor, VideoMAEModel)

_shared = {}
def get_audio_video():
    if "audio" not in _shared:
        print("Carregando áudio (emoção) e vídeo (VideoMAE)...")
        _shared["afe"] = AutoFeatureExtractor.from_pretrained(AUDIO_MODEL)
        _shared["audio"] = Wav2Vec2ForSequenceClassification.from_pretrained(AUDIO_MODEL).to(DEVICE).eval()
        _shared["vproc"] = VideoMAEImageProcessor.from_pretrained(VIDEO_MODEL)
        _shared["video"] = VideoMAEModel.from_pretrained(VIDEO_MODEL).to(DEVICE).eval()
        for m in (_shared["audio"], _shared["video"]):
            for p in m.parameters(): p.requires_grad = False
    return _shared

_text = {}
def get_text_model(lang):
    if lang not in _text:
        name = LANG_CFG[lang]["text_model"]
        print(f"[{lang}] carregando texto: {name}")
        tok = AutoTokenizer.from_pretrained(name)
        mdl = AutoModelForSequenceClassification.from_pretrained(name).to(DEVICE).eval()
        for p in mdl.parameters(): p.requires_grad = False
        _text[lang] = (tok, mdl)
    return _text[lang]

def _stats(seq):  # seq: [1, T, D] -> [2D] (mean ⊕ std)
    mean = seq.mean(dim=1)
    std = seq.std(dim=1) if seq.shape[1] > 1 else torch.zeros_like(mean)
    return torch.cat([mean, std], dim=-1).squeeze(0).float().cpu()

@torch.no_grad()
def text_features(txt, lang):
    tok, mdl = get_text_model(lang)
    if not txt.strip(): txt = "."
    inp = tok(txt, return_tensors="pt", truncation=True, max_length=128, padding=True).to(DEVICE)
    out = mdl(**inp, output_hidden_states=True)
    stats = _stats(out.hidden_states[-1])                       # [2*hidden]
    probs = out.logits.softmax(dim=-1).squeeze(0).float().cpu() # [n_classes do encoder]
    return torch.cat([stats, probs], dim=-1)

@torch.no_grad()
def audio_features(wav_path):
    s = get_audio_video()
    speech, _ = sf.read(wav_path)
    if getattr(speech, "ndim", 1) > 1: speech = speech.mean(axis=1)
    inp = s["afe"](speech, sampling_rate=16000, return_tensors="pt")
    iv = inp["input_values"].to(DEVICE)
    hidden = s["audio"].wav2vec2(iv).last_hidden_state              # [1, T, 768]
    return _stats(hidden)

@torch.no_grad()
def image_features(faces_dir):
    s = get_audio_video()
    paths = sorted(glob.glob(f"{faces_dir}/*.jpg") + glob.glob(f"{faces_dir}/*.png"))
    if not paths: raise FileNotFoundError(faces_dir)
    if len(paths) != N_FRAMES:
        idx = [int(i * len(paths) / N_FRAMES) for i in range(N_FRAMES)]
        paths = [paths[i] for i in idx]
    imgs = [Image.open(p).convert("RGB") for p in paths]
    inp = s["vproc"](imgs, return_tensors="pt").to(DEVICE)
    hidden = s["video"](**inp).last_hidden_state                   # [1, ~1568, 768]
    return _stats(hidden)

## 6. Gera o cache v2

In [7]:
def generate_cache(df, lang):
    out_dir = cache_dir(lang); os.makedirs(out_dir, exist_ok=True)
    n_ok, n_skip, n_err = 0, 0, 0
    for r in tqdm(df.itertuples(), total=len(df), desc=f"[{lang}] cache v2"):
        dst = f"{out_dir}/{r.sample_id}.pt"
        if os.path.exists(dst): n_skip += 1; continue
        try:
            txt_path = f"{processed_dir(lang, 'transcripts')}/{r.sample_id}.txt"
            wav_path = f"{processed_dir(lang, 'audio')}/{r.sample_id}.wav"
            faces = f"{processed_dir(lang, 'faces')}/{r.sample_id}"
            txt = open(txt_path, encoding="utf-8").read() if os.path.exists(txt_path) else ""
            torch.save({
                "text":  text_features(txt, lang),
                "audio": audio_features(wav_path),
                "image": image_features(faces),
                "label": torch.tensor(r.label_id, dtype=torch.long),
            }, dst)
            n_ok += 1
        except Exception as e:
            n_err += 1
            print(f"[erro] {r.sample_id}: {type(e).__name__}: {e}")
    print(f"[{lang}] ok={n_ok} skip={n_skip} err={n_err}")

def process_language(lang):
    print(f"\n{'='*60}\n>> Idioma: {lang}\n{'='*60}")
    df = build_dataframe(lang)
    run_audio_extraction(df, lang)
    run_transcription(df, lang)
    run_faces(df, lang)
    generate_cache(df, lang)

for lang in LANGS_TO_RUN:
    process_language(lang)


>> Idioma: en
[en] 600 vídeos | por classe: {'Anger': 84, 'Disgust': 85, 'Fear': 84, 'Happy': 85, 'Neutral': 93, 'Sadness': 79, 'Surprised': 90}


[en] cache v2:   0%|          | 0/600 [00:00<?, ?it/s]

[en] carregando texto: j-hartmann/emotion-english-distilroberta-base
Carregando áudio (emoção) e vídeo (VideoMAE)...


c:\Users\luis-\Documents\codigos\academico\MultimodalEmotionReconizer\.venv-torch\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\luis-\.cache\huggingface\hub\models--superb--wav2vec2-base-superb-er. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\luis-\Documents\codigos\academico\MultimodalEmo

[en] ok=600 skip=0 err=0

>> Idioma: pt
[pt] 604 vídeos | por classe: {'Anger': 84, 'Disgust': 85, 'Fear': 88, 'Happy': 85, 'Neutral': 93, 'Sadness': 79, 'Surprised': 90}


[pt] transcrição:   0%|          | 0/604 [00:00<?, ?it/s]Device set to use cpu
Using cache found in C:\Users\luis-/.cache\torch\hub\snakers4_silero-vad_master
[pt] transcrição:   0%|          | 1/604 [00:05<59:31,  5.92s/it]c:\Users\luis-\Documents\codigos\academico\MultimodalEmotionReconizer\.venv-torch\Lib\site-packages\transformers\models\whisper\generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, None], [2, 50359]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[pt] cache v2:   0%|          | 0/604 [00:00<?, ?it/s]

[pt] carregando texto: cardiffnlp/twitter-xlm-roberta-base-sentiment


c:\Users\luis-\Documents\codigos\academico\MultimodalEmotionReconizer\.venv-torch\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\luis-\.cache\huggingface\hub\models--cardiffnlp--twitter-xlm-roberta-base-sentiment. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
[pt] cache v2: 100%|██████████| 604/604 [

[pt] ok=604 skip=0 err=0


## 7. Sanidade — confere shapes do cache gerado

In [8]:
for lang in LANGS_TO_RUN:
    files = sorted(glob.glob(f"{cache_dir(lang)}/*.pt"))
    print(f"\n[{lang}] {len(files)} amostras em {cache_dir(lang)}")
    if files:
        s = torch.load(files[0], weights_only=False)
        for k, v in s.items():
            print(f"   {k}: {tuple(v.shape) if hasattr(v,'shape') else v}",
                  f"{v.dtype if hasattr(v,'dtype') else ''}")


[en] 600 amostras em ../cache_en_v2/multimodal
   text: (1543,) torch.float32
   audio: (1536,) torch.float32
   image: (1536,) torch.float32
   label: () torch.int64

[pt] 604 amostras em ../cache_pt_v2/multimodal
   text: (1539,) torch.float32
   audio: (1536,) torch.float32
   image: (1536,) torch.float32
   label: () torch.int64
